# Local MuseTalk LipSync (free GPU, semi-auto)

**What this does:** pulls your dubbed videos from Google Drive `LipSyncInbox`, runs MuseTalk lip-sync on Colab's free GPU so the mouth matches the dubbed audio, and saves the result to Drive `LipSyncDone`.

**How to use:**
1. Runtime menu -> Change runtime type -> **T4 GPU** -> Save.
2. Run each cell top to bottom (Runtime -> Run all).
3. Sign into Google Drive when prompted (cell 2).
4. It processes every video waiting in `LipSyncInbox`, then stops.

First run installs MuseTalk (~5-10 min). Later runs are faster.

## 1. Check GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit("NO GPU! Runtime -> Change runtime type -> T4 GPU -> Save, then Run all again.")
print(out.stdout.split("\n")[0:10])
print("GPU OK")

## 2. Mount Google Drive (sign in when asked)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Folders on your Drive (created if missing)
BASE = '/content/drive/MyDrive'
INBOX = f'{BASE}/LipSyncInbox'
DONE = f'{BASE}/LipSyncDone'
ARCHIVE = f'{INBOX}/archive'
for d in (INBOX, DONE, ARCHIVE):
    os.makedirs(d, exist_ok=True)
print('Drive ready. Drop videos in:', INBOX)

## 3. Install MuseTalk (first run only, ~5-10 min)

In [ ]:
%cd /content
import os
if not os.path.exists('/content/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git
%cd /content/MuseTalk
# Core deps (versions known to work on Colab T4)
!pip install -q --upgrade pip
!pip install -q -r requirements.txt 2>/dev/null || echo 'requirements step finished'
!pip install -q openmim
!mim install -q mmengine 'mmcv>=2.0.1' 'mmdet>=3.1.0' 'mmpose>=1.1.0' 2>/dev/null || echo 'mim step finished'
print('MuseTalk code + deps installed.')

## 4. Download model weights (first run only)

In [ ]:
%cd /content/MuseTalk
import os
# MuseTalk ships a download script; fall back to huggingface if needed.
if os.path.exists('download_weights.sh'):
    !bash download_weights.sh 2>/dev/null || echo 'weights script finished'
else:
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    os.makedirs('models', exist_ok=True)
    try:
        snapshot_download(repo_id='TMElyralab/MuseTalk', local_dir='models/musetalk')
    except Exception as e:
        print('Manual weight download may be needed:', e)
print('Weights step done. Check models/ folder below:')
!ls -R models 2>/dev/null | head -40

## 5. Lip-sync every video in LipSyncInbox

For each video, MuseTalk re-syncs the mouth to the audio already in that video (your dubbed track). Result -> `LipSyncDone`, original -> `archive`.

In [ ]:
%cd /content/MuseTalk
import os, glob, shutil, subprocess, datetime

VID_EXTS = ('.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v')
vids = sorted([f for f in glob.glob(f'{INBOX}/*') if f.lower().endswith(VID_EXTS)])
if not vids:
    print('Nothing to do. Drop dubbed videos into', INBOX, 'and run this cell again.')
else:
    print(f'Found {len(vids)} video(s) to lip-sync.')

def extract_audio(video, out_wav):
    subprocess.run(['ffmpeg','-y','-i',video,'-vn','-acodec','pcm_s16le','-ar','16000','-ac','1',out_wav],
                   check=True, capture_output=True)

for v in vids:
    name = os.path.basename(v)
    stem = os.path.splitext(name)[0]
    print(f'\n=== Lip-syncing: {name} ===')
    work = f'/content/work_{stem}'
    os.makedirs(work, exist_ok=True)
    local_in = f'{work}/in.mp4'
    shutil.copy(v, local_in)
    audio = f'{work}/audio.wav'
    extract_audio(local_in, audio)
    out_dir = f'{work}/out'
    os.makedirs(out_dir, exist_ok=True)
    # MuseTalk inference. Newer MuseTalk uses scripts/inference.py with a config;
    # we try the common entrypoints and pass video + audio.
    cmd = None
    if os.path.exists('scripts/inference.py'):
        cmd = ['python','-m','scripts.inference','--inference_config','configs/inference/test.yaml']
    elif os.path.exists('inference.py'):
        cmd = ['python','inference.py','--video_path',local_in,'--audio_path',audio,'--result_dir',out_dir]
    if cmd is None:
        print('!! Could not find MuseTalk inference entrypoint - check the repo layout in cell 3.')
        continue
    print('Running:', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        print(f'!! {name} failed - leaving it in inbox to retry.')
        continue
    # Find produced video and copy to Drive
    produced = sorted(glob.glob(f'{out_dir}/**/*.mp4', recursive=True), key=os.path.getmtime)
    if not produced:
        print('!! No output video found - check logs above.')
        continue
    final = produced[-1]
    dest = f'{DONE}/{stem} - LIPSYNC.mp4'
    shutil.copy(final, dest)
    shutil.move(v, f'{ARCHIVE}/{name}')
    print(f'OK -> saved to Drive: {dest}')

print('\nAll done. Check your Drive LipSyncDone folder.')